In [12]:
import random
from cgra import *
from kernels import *

In [13]:
kernel_name = "benchmarks/compigra_blas_paper/satilp/mmul_batch/3x3/"
version = "_3_IJK24"

In [14]:
# Global variables
CGRA_N_ROWS = 3
CGRA_N_COLS = 3
# Adress
first_addr = 20000

In [15]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [16]:
# Data
def configMemory(A_data, B_data, C_data, rowsA, colsA, colsB, n_batch):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # Config values
    # ----------------------            
    first_addr_A = first_addr
    first_addr_B = first_addr_A + n_batch*rowsA*colsA*4
    first_addr_C = first_addr_B + n_batch*colsA*colsB*4

    config_vals = [[] for i in range(CGRA_N_COLS)]

    if version == "_3_IJK24":
        config_vals[0] = [first_addr_C]
        config_vals[1] = [first_addr_A]
        config_vals[2] = [first_addr_B]
    
    if version == "_3_IJK60":
        config_vals[0] = [first_addr_C]
        config_vals[1] = [first_addr_A]
        config_vals[2] = [first_addr_B]
    
    addr_config_loads = [0 for i in range(CGRA_N_COLS)]
    for i in range(CGRA_N_COLS):
        kernel_add_memory_region(kernel_name, addr_config_loads[i], config_vals[i], version=version)
        if i < CGRA_N_COLS -1:
            addr_config_loads[i+1] = addr_config_loads[i] + len(config_vals[i])*4
    # Load data
    kernel_add_memory_region(kernel_name, first_addr_A, A_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_B, B_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_C, C_data, version=version)
    # Config data address for direct loads
    return addr_config_loads

In [17]:
def runKernel(load_addrs, max_it=1000, pr=["ROUT","INST"], printVal=1):
    # Run kernel
    run(kernel_name, pr=pr, load_addrs=load_addrs, version=version, limit=max_it, printVal=printVal)

In [18]:
def getResult(first_addr_C, end_addr_C, sizeRes):
    result = [0 for _ in range(sizeRes)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result[int((int(row[0]) - first_addr_C)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [19]:
def mmul_batch_cpu(A_data, B_data, rowsA, colsA, colsB, n_batch):
    expected_res = expected_res = [0 for _ in range(n_batch * rowsA * colsB)]
    for b in range (n_batch):
        for rA in range(rowsA):
            for cB in range(colsB):
                sum = 0
                for cA in range(colsA):
                    idxA = b * (rowsA * colsA) + rA * colsA + cA
                    idxB = b * (colsA * colsB) + cA * colsB + cB
                    sum += A_data[idxA] * B_data[idxB]
                idxOut = b * (rowsA * colsB) + rA * colsB + cB
                expected_res[idxOut] = sum
    return expected_res

In [20]:
# Test dimensions (4xXx4)
rowsA = 24
colsA = 24
colsB = 24
n_batch = 4

A_data = [random.randint(-10, 10) for _ in range(n_batch * rowsA * colsA)]
B_data = [random.randint(-10, 10) for _ in range(n_batch * colsA * colsB)]
C_data = [random.randint(-10, 10) for _ in range(n_batch * rowsA * colsB)]

A_data_cpy = A_data.copy()
B_data_cpy = B_data.copy()
C_data_cpy = C_data.copy()

load_addrs = configMemory(A_data, B_data, C_data, rowsA, colsA, colsB, n_batch)

In [ ]:
runKernel(load_addrs, max_it=20000000, printVal=0)

In [ ]:
# Get result from CGRA
first_addr_C = first_addr + n_batch*rowsA*colsA*4 + n_batch*colsA*colsB*4
result = getResult(first_addr_C, first_addr_C + n_batch*rowsA*colsB*4, n_batch*rowsA*colsB)

# Get cpu output
expected_res = mmul_batch_cpu(A_data_cpy, B_data_cpy, rowsA, colsA, colsB, n_batch)

# Check result correctness
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != result[i]:
        errors += 1
if errors > 0:
    print("Err: " + str(errors))
    print("Expected: ")
    printAsMatrix(expected_res, rowsA, colsB)
    print("CGRA: ")
    printAsMatrix(result, rowsA, colsB)
else:
    print("OK")

